In [2]:
# show → code input visible by default
# hide-output → output hidden by default
# show hide-output → both (can combine on one line)

import pandas as pd
#from google.cloud import bigquery
from common_lib.sql import BigQueryConnector
from common_lib.export import export_notebook_html
import os


## Get Data

In [3]:
# read csv file with neon transaction data
neon_data = pd.read_csv('./data/transaction-report-2026-07-08-15-46-23.csv')
neon_data

,Status,Dispute Status,Items Subtotal,Total Excluding Tax,Subtotal,Taxes,Total,FX Rate,Fee Amount,Net Proceeds,...,Account Display Name,Account ID,Order Number,Date,SKUs,Items,Property Display Name,Property ID,Environment Display Name,Environment ID
0,succeeded,NaN,0.99,0.99,0.99,0.00,0.99,1.0,0.05,0.94,...,TM-NKPTUXKKJOIPJQPN,5781706122FCB725,4T3F-VX6T-8LT9,2026-07-08T15:40:48.045Z,OfferTrack-TimedAlbum-IncPack-4P_099Bundle (qt...,1x Offer (OfferTrack-TimedAlbum-IncPack-4P_099...,Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617
1,succeeded,NaN,0.99,0.99,0.99,0.00,0.99,1.0,0.05,0.94,...,TM-KPNJRJPQXURSPKPR,9727A9CF87191572,PF9V-8B7P-58JF,2026-07-08T15:40:20.426Z,OfferTrack-TimedAlbum-IncPack-4P_099Bundle (qt...,1x Offer (OfferTrack-TimedAlbum-IncPack-4P_099...,Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617
2,succeeded,NaN,2.99,2.99,2.99,0.23,3.22,1.0,0.15,2.84,...,TM-IILUMPXOSNTMSRXN,5F9A4B5A6F74C300,J4WZ-26DX-92KD,2026-07-08T15:36:44.877Z,Feature-TipJar (qty: 1),1x Tip Jar (Feature-TipJar),Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617
3,succeeded,NaN,1.99,1.99,1.99,0.15,2.14,1.0,0.10,1.89,...,TM-SPLVVUPPJSVRRPLK,23799DA177CDD37A,56C8-Q64W-6LPV,2026-07-08T15:36:16.743Z,OfferTrack-TimedAlbum-IncPack-8P_199Bundle (qt...,1x Offer (OfferTrack-TimedAlbum-IncPack-8P_199...,Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617
4,succeeded,NaN,0.99,0.99,0.99,0.00,0.99,1.0,0.05,0.94,...,TM-KXVLIINNLMSQPRJU,C1978A4355003DF2,5F9M-R4MP-TPP2,2026-07-08T15:33:45.587Z,OfferTrack-TimedAlbum-IncPack-4P_099Bundle (qt...,1x Offer (OfferTrack-TimedAlbum-IncPack-4P_099...,Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74304,expired,NaN,1.99,1.99,1.99,NaN,NaN,NaN,NaN,NaN,...,TM-OLWONIMTMKVOJMUQ,8C416D24B4056E36,NaN,2026-02-16T11:57:13.269Z,bank-gems-gems_1 (qty: 1),1x Small amount of gems (bank-gems-gems_1),Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617
74305,expired,NaN,1.99,1.99,1.99,NaN,NaN,NaN,NaN,NaN,...,TM-OLWONIMTMKVOJMUQ,8C416D24B4056E36,NaN,2026-02-16T09:41:49.456Z,bank-gems-gems_1 (qty: 1),1x Small amount of gems (bank-gems-gems_1),Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617
74306,expired,NaN,1.99,1.99,1.99,NaN,NaN,NaN,NaN,NaN,...,TM-OLWONIMTMKVOJMUQ,8C416D24B4056E36,NaN,2026-02-16T09:35:00.550Z,bank-gems-gems_1 (qty: 1),1x Small amount of gems (bank-gems-gems_1),Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617
74307,expired,NaN,4.99,4.99,4.99,NaN,NaN,NaN,NaN,NaN,...,TM-OLWONIMTMKVOJMUQ,8C416D24B4056E36,NaN,2026-02-16T09:29:33.457Z,bank-gems-gems_2 (qty: 1),1x Bescheidene Anzahl Edelsteine (bank-gems-ge...,Love & Pies,63992876-766b-4a00-86c0-8cb44ac6d35c,Production,dd390cfe-756e-4c59-8adc-8a309a95b617


In [4]:
refresh_data = True

In [5]:
query_location = './sql/neonpay_rp_IAPSuccess.sql'
parameters = {
}

bqc = BigQueryConnector()
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 2.60 GB when run.
Estimated query cost: $0.02


In [6]:
omni_rp_data = pd.DataFrame()
if refresh_data:
    omni_rp_data = bqc.get(query='./sql/neonpay_rp_IAPSuccess.sql', is_path=True, query_parameters=parameters)
    omni_rp_data.to_pickle('./data/neonpay_rp_IAPSuccess.pkl')

omni_rp_data = pd.read_pickle('./data/neonpay_rp_IAPSuccess.pkl')

In [7]:
omni_rp_data

,event_ts,user_id,purchase_id,transaction_id,transaction_store,outcome,iap_price_usd,currency_code,currency_amount
0,2026-07-14 20:10:45.266082,F6927B08F3C82518,b8cc3f50-0e34-4fcb-b26f-3d10bf480b6d,1335974f-4395-4da1-b7ee-df524da4ef5b,NeonPay,completed,29.99,USD,29.99
1,2026-07-14 15:39:54.680837,F6927B08F3C82518,d3ed4608-f02e-4ded-8fc8-5c2f6b2ea279,49b6e1a4-50bb-4864-bdf7-03f3371c0c96,NeonPay,completed,34.99,USD,34.99
2,2026-07-14 06:02:11.567019,91E02417BFA42438,a7a8c5cb-d858-457e-a8e1-28946e8d0c44,1ed106d3-a146-421c-9bfa-240cad422d11,NeonPay,completed,34.99,USD,34.99
3,2026-07-14 13:59:38.931006,CF37D7B8A14DA265,6c0ff7a0-2d59-4221-81ae-a3a50ae21c17,e6f4eedb-d593-4a44-95fe-791a1cdc2b97,NeonPay,completed,34.99,USD,34.99
4,2026-07-14 01:31:14.902884,64130F15C1E41E55,ac0c324c-bb6a-40ed-9d63-49cc412003a0,b9e30f3f-0165-4f6a-9e1b-9f206674f433,NeonPay,completed,34.99,USD,34.99
...,...,...,...,...,...,...,...,...,...
67054,2026-03-19 16:23:23.048316,9397623F58CEA581,9172b515-4646-4f1c-b8f6-f0e604f4204d,2f8432b0-d804-44dc-b0a0-a7386752e54a,NeonPay,completed,1.99,USD,1.99
67055,2026-03-19 02:11:19.673972,3EF854E58E0EBCA1,ee792318-a0a0-40d8-9091-0ae2616d5e3b,8e355c9e-874c-45ba-b6f7-435b05e55e25,NeonPay,completed,1.99,USD,1.99
67056,2026-03-19 23:05:00.663327,9397623F58CEA581,2bb2a542-ef3e-4752-83e2-48edeef4eef5,515faf28-aa17-4685-b0a5-3f3a8911b649,NeonPay,completed,1.99,USD,1.99
67057,2026-03-19 19:12:32.652590,243631BA2B7791C8,54e790fd-358f-4c4d-a1f8-169b4e9ffe4f,582811aa-0fbb-4b19-8bc3-0b645595d00c,NeonPay,completed,4.99,USD,4.99


In [8]:
query_location = './sql/neonpay_fact_IAPProduct.sql'
parameters = {
}

bqc = BigQueryConnector()
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 183.3 MB when run.
Estimated query cost: $0.00


In [9]:
omni_product_data = pd.DataFrame()
if refresh_data:
    omni_product_data = bqc.get(query='./sql/neonpay_fact_IAPProduct.sql', is_path=True, query_parameters=parameters)
    omni_product_data.to_pickle('./data/neonpay_fact_IAPProduct_data.pkl')

omni_product_data = pd.read_pickle('./data/neonpay_fact_IAPProduct_data.pkl')

In [10]:
omni_product_data

,user_id,transaction_store,product_category,product_type,product_theme,product_id,product_price_group,iap_price_usd,dt,usd_iap_revenue,usd_net_iap_revenue,usd_iap_price_revenue,usd_net_iap_price_revenue,n_trans,loading_timestamp
0,21B578A30B74D016,NeonPay,gems,MiniShop,Gems,MiniShop-Gems-Gems1,gems_0099_0499,1.99,2026-05-14,3.98,3.7810,3.98,3.7810,2,2026-05-19 02:23:52.357046+00:00
1,6825E202BFCAA26,NeonPay,gems,MiniShop,Gems,MiniShop-Gems-Gems1,gems_0099_0499,1.99,2026-05-14,1.99,1.8905,1.99,1.8905,1,2026-05-19 02:23:52.357046+00:00
2,961383D4464E62B9,NeonPay,gems,MiniShop,Gems,MiniShop-Gems-Gems1,gems_0099_0499,1.99,2026-05-14,1.99,1.8905,1.99,1.8905,1,2026-05-19 02:23:52.357046+00:00
3,B80ADA3ACCFEC958,NeonPay,gems,MiniShop,Gems,MiniShop-Gems-Gems1,gems_0099_0499,1.99,2026-05-14,1.99,1.8905,1.99,1.8905,1,2026-05-19 02:23:52.357046+00:00
4,D816BD6258CECDC5,NeonPay,gems,MiniShop,Gems,MiniShop-Gems-Gems1,gems_0099_0499,1.99,2026-05-14,1.99,1.8905,1.99,1.8905,1,2026-05-19 02:23:52.357046+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65572,F5DA07F3174E3AC5,NeonPay,gems,Bank,Gems,bank-gems-gems_4,gems_1099_1999,19.99,2026-05-22,19.99,18.9905,19.99,18.9905,1,2026-05-27 02:43:34.722199+00:00
65573,8F243C80733EE74E,NeonPay,gems,Bank,Gems,bank-gems-gems_4,gems_1099_1999,19.99,2026-05-22,19.99,18.9905,19.99,18.9905,1,2026-05-27 02:43:34.722199+00:00
65574,8F134E18B224BCAA,NeonPay,gems,Bank,Gems,bank-gems-gems_5,gems_2099_4999,49.99,2026-05-22,49.99,47.4905,49.99,47.4905,1,2026-05-27 02:43:34.722199+00:00
65575,6F9413C0BB3F71DA,NeonPay,gems,Bank,Gems,bank-gems-gems_5,gems_2099_4999,49.99,2026-05-22,49.99,47.4905,49.99,47.4905,1,2026-05-27 02:43:34.722199+00:00


In [11]:
query_location = './sql/neonpay_fact_PurchaseRates.sql'
parameters = {
}

bqc = BigQueryConnector()
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 28.0 MB when run.
Estimated query cost: $0.00


In [12]:
omni_purchase_data = pd.DataFrame()
if refresh_data:
    omni_purchase_data = bqc.get(query='./sql/neonpay_fact_PurchaseRates.sql', is_path=True, query_parameters=parameters)
    omni_purchase_data.to_pickle('./data/neonpay_fact_PurchaseRates_data.pkl')

omni_purchase_data = pd.read_pickle('./data/neonpay_fact_PurchaseRates_data.pkl')

In [13]:
omni_purchase_data

,user_id,dt,product_id,product_category,price,n_offered_dtc,purchased,purchased_dtc,n_trans_dtc,n_trans,usd_iap_price_revenue_dtc,usd_iap_price_revenue,usd_net_iap_price_revenue_dtc,usd_net_iap_price_revenue,loading_timestamp
0,EC5A0AD3A796A487,2026-04-04,MiniShop-Gems-Gems1,gems,1.99,1,1,1,1,1,1.99,1.99,1.8905,1.8905,2026-04-09 02:23:53.913034+00:00
1,A6F4E3AD34D0A600,2026-04-04,MiniShop-Gems-Gems1,gems,1.99,1,1,1,1,1,1.99,1.99,1.8905,1.8905,2026-04-09 02:23:53.913034+00:00
2,F16798EF61ECDF0D,2026-04-04,MiniShop-Gems-Gems1,gems,1.99,1,1,1,1,1,1.99,1.99,1.8905,1.8905,2026-04-09 02:23:53.913034+00:00
3,7CE55ED3F106E7EA,2026-04-04,MiniShop-Gems-Gems1,gems,1.99,1,1,1,1,1,1.99,1.99,1.8905,1.8905,2026-04-09 02:23:53.913034+00:00
4,A6C0D8C117879DC8,2026-04-04,MiniShop-Gems-Gems1,gems,1.99,1,1,1,1,1,1.99,1.99,1.8905,1.8905,2026-04-09 02:23:53.913034+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58531,441AA059644A17EF,2026-05-23,bank-gems-gems_4,gems,19.99,1,1,1,1,1,19.99,19.99,18.9905,18.9905,2026-05-28 03:08:28.790353+00:00
58532,5B4EB6318E917AE1,2026-05-23,bank-gems-gems_4,gems,19.99,1,1,1,1,1,19.99,19.99,18.9905,18.9905,2026-05-28 03:08:28.790353+00:00
58533,1A0A0782598C6B0F,2026-05-23,bank-gems-gems_4,gems,19.99,1,1,1,1,1,19.99,19.99,18.9905,18.9905,2026-05-28 03:08:28.790353+00:00
58534,362F19627BB17ECA,2026-05-23,bank-gems-gems_6,gems,99.99,1,1,1,1,1,99.99,99.99,94.9905,94.9905,2026-05-28 03:08:28.790353+00:00


## Check data

In [14]:
neon_data.groupby(['Currency']).size().reset_index(name='count').sort_values(by='count', ascending=False)

,Currency,count
7,USD,74248
4,GBP,43
2,CNY,6
3,EUR,5
1,CAD,4
0,AUD,1
5,NOK,1
6,PEN,1


In [15]:
omni_rp_data.groupby(['currency_code']).size().reset_index(name='count').sort_values(by='count', ascending=False)

,currency_code,count
7,USD,66910
3,GBP,105
5,JPY,26
1,CAD,7
2,EUR,6
6,NOK,3
0,AUD,1
4,INR,1


In [16]:
neon_data.groupby(['Status']).size().reset_index(name='count').sort_values(by='count', ascending=False)

,Status,count
5,succeeded,62528
1,expired,9790
2,failed,1855
3,incomplete,85
0,disputed,38
4,refunded,13


In [17]:
omni_rp_data.groupby(['outcome']).size().reset_index(name='count').sort_values(by='count', ascending=False)

,outcome,count
0,completed,67059


## Process data

In [27]:
neon_data_fixed = pd.DataFrame()
neon_data_fixed = neon_data

neon_data_fixed['Date'] = pd.to_datetime(neon_data_fixed['Date'])
neon_data_fixed['Date_trunc'] = pd.to_datetime(neon_data_fixed['Date']).dt.date
neon_data_fixed['Date_month'] = pd.to_datetime(neon_data_fixed['Date']).dt.to_period('M')


neon_data_fixed = neon_data_fixed[neon_data_fixed.Status=='succeeded']
neon_data_fixed = neon_data_fixed[(neon_data_fixed.Date_trunc >= pd.to_datetime('2026-05-01').date()) & (neon_data_fixed.Date_trunc < pd.to_datetime('2026-07-01').date())]
neon_data_fixed.sort_values(by='Date', ascending=True, inplace=True)
neon_data_fixed[['Date','Date_trunc','Date_month','Account ID','Items Subtotal','Total Excluding Tax','Subtotal','Fee Amount','Net Proceeds','Currency']]

/var/folders/vj/nwmxdwwn0hl5rtj_xcywvvpc0000gn/T/ipykernel_94988/26821495.py:6: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  neon_data_fixed['Date_month'] = pd.to_datetime(neon_data_fixed['Date']).dt.to_period('M')


,Date,Date_trunc,Date_month,Account ID,Items Subtotal,Total Excluding Tax,Subtotal,Fee Amount,Net Proceeds,Currency
43846,2026-05-01 00:00:53.608000+00:00,2026-05-01,2026-05,DAE598C198F57109,6.99,6.99,6.99,0.35,6.64,USD
43845,2026-05-01 00:04:07.247000+00:00,2026-05-01,2026-05,E8E6CA67C07B945,6.99,6.99,6.99,0.35,6.64,USD
43844,2026-05-01 00:05:22.955000+00:00,2026-05-01,2026-05,EF449C1A73AD4ACB,19.99,19.99,19.99,1.00,18.99,USD
43842,2026-05-01 00:05:30.936000+00:00,2026-05-01,2026-05,48E2C02270F7B99A,19.99,19.99,19.99,1.00,18.99,USD
43841,2026-05-01 00:08:10.213000+00:00,2026-05-01,2026-05,8BE232F34550A513,24.99,24.99,24.99,1.25,23.74,USD
...,...,...,...,...,...,...,...,...,...,...
5796,2026-06-30 23:53:18.538000+00:00,2026-06-30,2026-06,8DB77FC1C02E8ECF,9.99,9.99,9.99,0.50,9.49,USD
5795,2026-06-30 23:53:31.315000+00:00,2026-06-30,2026-06,82EDAA0F666A4D21,2.99,2.99,2.99,0.15,2.84,USD
5794,2026-06-30 23:53:49.927000+00:00,2026-06-30,2026-06,54BBDF507E8F9B30,19.99,19.99,19.99,1.00,18.99,USD
5793,2026-06-30 23:54:46.604000+00:00,2026-06-30,2026-06,5C198FBAE635DFCC,0.99,0.99,0.99,0.05,0.94,USD


In [28]:
test_user_day = neon_data_fixed.groupby(['Account ID','Date_trunc','Items Subtotal']).size().reset_index(name='count').sort_values(by='count', ascending=False)
test_user_day[test_user_day['count']>1]

,Account ID,Date_trunc,Items Subtotal,count
10584,647FA14B8BCEDFFC,2026-06-15,2.99,5
11844,6C68FCA58F8E0308,2026-06-15,1.99,5
29053,ED2E47BD0C5BB5FD,2026-05-01,1.99,5
10085,6162C5D3F95727A,2026-05-09,1.99,4
7030,47120DBBBD54AC37,2026-05-13,1.99,4
...,...,...,...,...
31124,FAD491AFDB17EF55,2026-05-27,1.99,2
2212,1F1E2DDBB8CCEEC3,2026-06-18,2.99,2
15593,8C5BE0B8406F9A4A,2026-05-27,1.99,2
4129,2CDAB2521182ADB8,2026-06-22,4.99,2


In [29]:
omni_rp_fixed = pd.DataFrame()
omni_rp_fixed = omni_rp_data

omni_rp_fixed['dt_trunc'] = pd.to_datetime(omni_rp_fixed['event_ts']).dt.date
omni_rp_fixed['dt_month'] = pd.to_datetime(omni_rp_fixed['event_ts']).dt.to_period('M')
omni_rp_fixed = omni_rp_fixed[omni_rp_fixed.transaction_store=='NeonPay']
omni_rp_fixed = omni_rp_fixed[(omni_rp_fixed.dt_trunc >= pd.to_datetime('2026-05-01').date()) & (omni_rp_fixed.dt_trunc < pd.to_datetime('2026-07-01').date())]
omni_rp_fixed.sort_values(by='event_ts', ascending=True, inplace=True)
omni_rp_fixed

,event_ts,user_id,purchase_id,transaction_id,transaction_store,outcome,iap_price_usd,currency_code,currency_amount,dt_trunc,dt_month
43906,2026-05-01 00:01:29.016066,DAE598C198F57109,74d66490-8db9-479e-802b-0dbb897fd17a,e92ce86b-6eda-44ae-a705-6e4ecfa798fe,NeonPay,completed,6.99,USD,6.99,2026-05-01,2026-05
43912,2026-05-01 00:05:00.841838,E8E6CA67C07B945,95133986-dcf3-45d5-ab79-9f79bb342947,33d874a4-87bb-45d3-a3c4-8ba1d0b9d5dc,NeonPay,completed,6.99,USD,6.99,2026-05-01,2026-05
44125,2026-05-01 00:06:29.704513,48E2C02270F7B99A,2307e71d-e0d3-4bb6-ac8d-ccfb0314ea62,4ae011f1-8f6b-41b6-b7cc-d3daec386a04,NeonPay,completed,19.99,USD,19.99,2026-05-01,2026-05
44114,2026-05-01 00:06:51.732503,EF449C1A73AD4ACB,d811e8e1-9e42-47ec-a359-e966102f7069,557a85bc-c8cb-44b3-b4c0-5fafbe9f7d73,NeonPay,completed,19.99,USD,19.99,2026-05-01,2026-05
44039,2026-05-01 00:09:45.610088,8BE232F34550A513,f11eece0-a4a6-42b2-b52a-96bb054ac7d2,ebb84433-aeff-4e15-b493-597574fafd29,NeonPay,completed,24.99,USD,24.99,2026-05-01,2026-05
...,...,...,...,...,...,...,...,...,...,...,...
2776,2026-06-30 23:54:23.454147,DFB687FB9EA41C4,c619fc85-4089-4b60-9703-c5cfd8ba3fcc,4cb12357-01a3-44c5-a147-5fd83478a064,NeonPay,completed,2.99,USD,2.99,2026-06-30,2026-06
3017,2026-06-30 23:54:25.746007,216AF3E5EEB4338C,d368395a-1569-4d21-a450-5b651a4c08ec,878802de-3c21-435b-af6a-60cb01a9b996,NeonPay,completed,1.99,USD,1.99,2026-06-30,2026-06
2812,2026-06-30 23:54:31.406614,54BBDF507E8F9B30,6e895c5a-a9c2-4fea-b9dc-eb7c1c97c521,897bcd25-52bc-4b8e-aa74-bc4527a7e765,NeonPay,completed,19.99,USD,19.99,2026-06-30,2026-06
3243,2026-06-30 23:55:46.983703,5C198FBAE635DFCC,13802e61-71a3-4d9f-a7b6-b8e565f9d420,e9953a82-3e21-4c61-91c7-c23ac6407e32,NeonPay,completed,0.99,USD,0.99,2026-06-30,2026-06


In [30]:
omni_product_fixed = pd.DataFrame()
omni_product_fixed = omni_product_data

omni_product_fixed['dt_month'] = pd.to_datetime(omni_product_fixed['dt']).dt.to_period('M')
#omni_product_fixed = omni_product_fixed[omni_product_fixed.transaction_store=='NeonPay']
omni_product_fixed = omni_product_fixed[(omni_product_fixed.dt >= pd.to_datetime('2026-05-01').date()) & (omni_product_fixed.dt < pd.to_datetime('2026-07-01').date())]
omni_product_fixed.sort_values(by='dt', ascending=True, inplace=True)
omni_product_fixed

/var/folders/vj/nwmxdwwn0hl5rtj_xcywvvpc0000gn/T/ipykernel_94988/1308150548.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  omni_product_fixed.sort_values(by='dt', ascending=True, inplace=True)


,user_id,transaction_store,product_category,product_type,product_theme,product_id,product_price_group,iap_price_usd,dt,usd_iap_revenue,usd_net_iap_revenue,usd_iap_price_revenue,usd_net_iap_price_revenue,n_trans,loading_timestamp,dt_month
56007,EC2629D3316C26B4,NeonPay,daily_offers,RotatingIncPack,RotatingOffersIncPack,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,daily_offers_1099_1999,12.99,2026-05-01,12.99,12.3405,12.99,12.3405,1,2026-05-06 01:59:20.224165+00:00,2026-05
55836,9CD16D137054409F,NeonPay,offer_track,TimedAlbumIncPack,none,OfferTrack-TimedAlbum-IncPack-8P_199Bundle,offer_track_0099_0499,1.99,2026-05-01,1.99,1.8905,1.99,1.8905,1,2026-05-06 01:59:20.224165+00:00,2026-05
55837,F1C8BE3ECBE32A36,NeonPay,offer_track,TimedAlbumIncPack,none,OfferTrack-TimedAlbum-IncPack-8P_199Bundle,offer_track_0099_0499,1.99,2026-05-01,1.99,1.8905,1.99,1.8905,1,2026-05-06 01:59:20.224165+00:00,2026-05
55838,BFA2D17B6B05B071,NeonPay,offer_track,TimedAlbumIncPack,none,OfferTrack-TimedAlbum-IncPack-8P_199Bundle,offer_track_0099_0499,1.99,2026-05-01,1.99,1.8905,1.99,1.8905,1,2026-05-06 01:59:20.224165+00:00,2026-05
55839,AD0260B2FFCEF110,NeonPay,offer_track,TimedAlbumIncPack,none,OfferTrack-TimedAlbum-IncPack-8P_199Bundle,offer_track_0099_0499,1.99,2026-05-01,1.99,1.8905,1.99,1.8905,1,2026-05-06 01:59:20.224165+00:00,2026-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16335,9654F3E82C0D6845,NeonPay,offer_track,TimedAlbumIncPack,none,OfferTrack-TimedAlbum-IncPack-8P_199Bundle,offer_track_0099_0499,1.99,2026-06-30,1.99,1.8905,1.99,1.8905,1,2026-07-05 02:23:38.574507+00:00,2026-06
16334,6F8E8AB04B9B69EE,NeonPay,offer_track,TimedAlbumIncPack,none,OfferTrack-TimedAlbum-IncPack-8P_199Bundle,offer_track_0099_0499,1.99,2026-06-30,1.99,1.8905,1.99,1.8905,1,2026-07-05 02:23:38.574507+00:00,2026-06
16333,C91C1147A1D8269B,NeonPay,offer_track,TimedAlbumIncPack,none,OfferTrack-TimedAlbum-IncPack-8P_199Bundle,offer_track_0099_0499,1.99,2026-06-30,1.99,1.8905,1.99,1.8905,1,2026-07-05 02:23:38.574507+00:00,2026-06
16340,B0D200E203621B1D,NeonPay,offer_track,TimedAlbumIncPack,none,OfferTrack-TimedAlbum-IncPack-8P_199Bundle,offer_track_0099_0499,1.99,2026-06-30,1.99,1.8905,1.99,1.8905,1,2026-07-05 02:23:38.574507+00:00,2026-06


In [31]:
omni_purchase_fixed = pd.DataFrame()
omni_purchase_fixed = omni_purchase_data

omni_purchase_fixed['dt_month'] = pd.to_datetime(omni_purchase_fixed['dt']).dt.to_period('M')
omni_purchase_fixed = omni_purchase_fixed[(omni_purchase_fixed.dt >= pd.to_datetime('2026-05-01').date()) & (omni_purchase_fixed.dt < pd.to_datetime('2026-07-01').date())]
omni_purchase_fixed.sort_values(by='dt', ascending=True, inplace=True)
omni_purchase_fixed

/var/folders/vj/nwmxdwwn0hl5rtj_xcywvvpc0000gn/T/ipykernel_94988/3314423834.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  omni_purchase_fixed.sort_values(by='dt', ascending=True, inplace=True)


,user_id,dt,product_id,product_category,price,n_offered_dtc,purchased,purchased_dtc,n_trans_dtc,n_trans,usd_iap_price_revenue_dtc,usd_iap_price_revenue,usd_net_iap_price_revenue_dtc,usd_net_iap_price_revenue,loading_timestamp,dt_month
33717,86E67EEBE2BE1B4F,2026-05-01,bank-gems-gems_4,gems,19.99,1,1,1,1,1,19.99,19.99,18.9905,18.9905,2026-05-06 02:15:42.808482+00:00,2026-05
33525,9FE5B55D5D91F257,2026-05-01,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,daily_offers,7.99,1,1,1,1,1,7.99,7.99,7.5905,7.5905,2026-05-06 02:15:42.808482+00:00,2026-05
33526,47C37B86F93B0789,2026-05-01,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,daily_offers,3.99,1,1,1,1,1,3.99,3.99,3.7905,3.7905,2026-05-06 02:15:42.808482+00:00,2026-05
33527,9911A346575CA63,2026-05-01,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,daily_offers,3.99,1,1,1,1,1,3.99,3.99,3.7905,3.7905,2026-05-06 02:15:42.808482+00:00,2026-05
33528,A0B801D0EC243DBC,2026-05-01,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,daily_offers,3.99,1,1,1,1,1,3.99,3.99,3.7905,3.7905,2026-05-06 02:15:42.808482+00:00,2026-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24026,A4A97349DFB8E858,2026-06-30,OfferTrack-TimedAlbum-IncPack-8P_199Bundle,offer_track,1.99,1,1,1,1,1,1.99,1.99,1.8905,1.8905,2026-07-05 02:28:12.668858+00:00,2026-06
24027,FCCF399F4008286F,2026-06-30,OfferTrack-TimedAlbum-IncPack-8P_199Bundle,offer_track,1.99,1,1,1,1,1,1.99,1.99,1.8905,1.8905,2026-07-05 02:28:12.668858+00:00,2026-06
24028,42A2C39E4A8871AF,2026-06-30,OfferTrack-TimedAlbum-IncPack-8P_199Bundle,offer_track,1.99,1,1,1,1,1,1.99,1.99,1.8905,1.8905,2026-07-05 02:28:12.668858+00:00,2026-06
24014,9B57EFB2CBEEF486,2026-06-30,OfferTrack-TimedAlbum-IncPack-8P_199Bundle,offer_track,1.99,1,1,1,1,1,1.99,1.99,1.8905,1.8905,2026-07-05 02:28:12.668858+00:00,2026-06


## Reconcile data

In [32]:
neon_data_recon = neon_data_fixed
neon_data_recon = neon_data_recon.groupby(['Date_month']).agg(
    iap_revenue = ('Items Subtotal', 'sum'),
    iap_net_revenue = ('Net Proceeds', 'sum')
)

neon_data_recon

,iap_revenue,iap_net_revenue
Date_month,,
2026-05,145609.81,138334.26
2026-06,130521.94,123988.66


In [33]:
omni_rp_recon = omni_rp_fixed
omni_rp_recon = omni_rp_recon.groupby(['dt_month']).agg(
    iap_revenue = ('iap_price_usd', 'sum')
)

omni_rp_recon['iap_net_revenue'] = omni_rp_recon['iap_revenue'] * 0.95

omni_rp_recon

,iap_revenue,iap_net_revenue
dt_month,,
2026-05,145678.83,138394.8885
2026-06,130514.94,123989.1930


In [34]:
omni_product_recon = omni_product_fixed
omni_product_recon = omni_product_recon.groupby(['dt_month']).agg(
    iap_revenue = ('usd_iap_price_revenue', 'sum'),
    iap_net_revenue = ('usd_net_iap_revenue', 'sum')
)

omni_product_recon

,iap_revenue,iap_net_revenue
dt_month,,
2026-05,145678.83,138583.522925
2026-06,130514.94,124395.158583


In [35]:
omni_purchase_recon = omni_purchase_fixed
omni_purchase_recon = omni_purchase_recon.groupby(['dt_month']).agg(
    iap_revenue = ('usd_iap_price_revenue_dtc', 'sum'),
    iap_net_revenue = ('usd_net_iap_price_revenue_dtc', 'sum')
)

omni_purchase_recon

,iap_revenue,iap_net_revenue
dt_month,,
2026-05,133346.32,126679.0040
2026-06,117253.87,111391.1765
